[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-01-polars-vs-pandas-the-mental-model-shift.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Polars vs Pandas: the Mental Model Shift
**certified-journeys / polars-certified** &nbsp;|&nbsp; Foundation

> **Goal for today:** Understand why Polars exists, how it differs from Pandas at a memory and execution level, and run your first Polars pipeline.

---
## Why Polars?

Pandas was built in 2008 for a world of single-core CPUs and gigabyte-scale data. Polars was built in 2020 for multi-core CPUs, lazy evaluation, and datasets that don't fit in RAM.

| Dimension | Pandas | Polars |
|---|---|---|
| Memory model | Row-oriented NumPy arrays | Column-oriented Apache Arrow |
| Execution | Eager only | Eager **and** Lazy (query optimizer) |
| API style | Index-based, mutable | Expression-based, immutable |
| Parallelism | Single-threaded by default | Multi-threaded by default |
| Null handling | `NaN` (float hack) | `null` — first-class, type-safe |
| Speed (typical) | Baseline | 5–20x faster on large data |

> **Key insight:** Polars is built on **Apache Arrow**. The column-oriented memory layout means entire columns live in contiguous memory — the CPU cache loves it. Understand this before optimizing.

In [ ]:
%pip install -q polars numpy pandas

---
## Step 1 · Create your first Polars DataFrame

The entry point is `pl.DataFrame` — it accepts dicts, lists, NumPy arrays, and Arrow tables.
Compare the syntax with Pandas: it looks similar, but the internals are completely different.

- Polars columns are **immutable Arrow arrays** — no in-place mutation
- There is **no index** — Polars deliberately removes the index concept
- Types are inferred at construction time and enforced strictly

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

data = {
    "product": ["apple", "banana", "cherry", "date"],
    "price":   [1.20,    0.50,     3.00,     2.50],
    "stock":   [100,     250,      80,       None],   # None becomes pl.Null
}

# --- Polars ---
df_pl = pl.DataFrame(data)
print("Polars DataFrame:")
print(df_pl)
print()

# --- Pandas equivalent (for comparison) ---
df_pd = pd.DataFrame(data)
print("Pandas DataFrame:")
print(df_pd)
print()

# Schema and shape — Polars is explicit about types
print("Polars schema:", df_pl.schema)
print("Polars shape: ", df_pl.shape)
print("Polars dtypes:", df_pl.dtypes)

**What just happened?**
- `None` in the `stock` column became a **typed null** (`Int64` with a null bitmask) — not `NaN`
- Polars printed the **schema** inline with the DataFrame — types are always visible
- There is **no row index** column — Polars removes this Pandas concept entirely
- **`df.schema`** returns a dict of `{col_name: dtype}` — use it to inspect before transforming
- Polars inferred `stock` as `Int64` (nullable integer), while Pandas would have used `float64` to encode `NaN`

---
## Step 2 · Null handling — Polars vs Pandas

Pandas uses `NaN` (a floating-point value) to represent missing data — this means integer columns get
silently promoted to float when a null appears. Polars uses **first-class nulls** backed by Arrow's
validity bitmask, so `Int64` stays `Int64` even with missing values.

| Operation | Pandas | Polars |
|---|---|---|
| Null value | `np.nan` (float) | `None` / `pl.Null` |
| Integer + null | Becomes `float64` | Stays `Int64` |
| Check nulls | `isna()` / `isnull()` | `is_null()` |
| Fill nulls | `fillna()` | `fill_null()` |

In [ ]:
# Demonstrate null handling differences
print("=== Polars null handling ===")
print("Null count per column:")
print(df_pl.null_count())          # how many nulls per column
print()

# is_null() returns a Boolean Series
print("stock is_null:", df_pl["stock"].is_null().to_list())

# fill_null: replace nulls with a scalar or strategy
df_filled = df_pl.with_columns(
    pl.col("stock").fill_null(0)   # replace null with 0
)
print("\nAfter fill_null(0):")
print(df_filled)

print("\n=== Pandas null: int becomes float ===")
print(df_pd.dtypes)                # stock becomes float64 due to NaN

**What just happened?**
- `df_pl.null_count()` returns a DataFrame with one row showing nulls per column — quick audit tool
- `fill_null()` is type-safe — filling an `Int64` column with `0` keeps the column as `Int64`
- **In Pandas, `stock` became `float64`** because `np.nan` is a float — a silent, error-prone promotion
- Polars' null system means you never accidentally lose integer precision due to missing data
- **`with_columns()`** returns a new DataFrame — Polars is always immutable

---
## Step 3 · Eager operations: filter, select, sort

Polars eager mode executes immediately, like Pandas. But the API is expression-based:
you never reference a column by `df["col"]` inside a transformation — you use `pl.col("col")`.

This separation lets Polars analyse and optimise operations before running them.

In [ ]:
# Build a slightly larger DataFrame to make operations meaningful
df = pl.DataFrame({
    "product":  ["apple", "banana", "cherry", "date", "elderberry", "fig"],
    "category": ["fruit",  "fruit",   "berry",   "fruit", "berry",     "fruit"],
    "price":    [1.20,     0.50,      3.00,      2.50,    4.80,        1.80],
    "stock":    [100,      250,       80,        None,    30,          200],
})

# --- filter: keep rows where price > 1.50 ---
cheap = df.filter(pl.col("price") > 1.50)
print("filter price > 1.50:")
print(cheap)
print()

# --- select: pick specific columns (returns a new DataFrame) ---
names_prices = df.select(pl.col("product"), pl.col("price"))
print("select product + price:")
print(names_prices)
print()

# --- sort: sort by price descending ---
top = df.sort("price", descending=True)
print("sort by price descending:")
print(top)

**What just happened?**
- `filter()` takes a **Polars expression** (`pl.col("price") > 1.50`), not a boolean Series — this is a key mental model shift
- `select()` returns only the requested columns — equivalent to Pandas `df[["product", "price"]]`
- `sort()` accepts a column name string or expression, and a `descending` flag
- **Every operation returns a new DataFrame** — Polars never mutates in place
- Notice there is no index in the output — row positions are implicit

---
## Step 4 · Load a CSV, group, aggregate, write

A real Polars pipeline: generate synthetic data, write a CSV, load it with `pl.read_csv()`,
filter, aggregate, and write results back. This is the core workflow you'll use every day.

In [ ]:
import random
import csv
import os

# --- Generate a synthetic CSV with 100 000 rows ---
random.seed(42)
categories = ["electronics", "clothing", "food", "books", "sports"]
regions    = ["north", "south", "east", "west"]

rows = [
    {
        "order_id":  i,
        "category":  random.choice(categories),
        "region":    random.choice(regions),
        "revenue":   round(random.uniform(5.0, 500.0), 2),
        "quantity":  random.randint(1, 20),
    }
    for i in range(1, 100_001)
]

csv_path = "/tmp/orders.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Written {len(rows):,} rows to {csv_path}")

In [ ]:
# --- Load with Polars ---
df = pl.read_csv(csv_path)          # reads into an eager DataFrame
print("Schema:", df.schema)
print("Shape: ", df.shape)
print()

# --- Filter: only electronics with revenue > 200 ---
electronics = df.filter(
    (pl.col("category") == "electronics") & (pl.col("revenue") > 200)
)
print(f"Electronics with revenue > 200: {len(electronics):,} rows")

# --- Group by category + region, aggregate ---
summary = (
    df
    .group_by(["category", "region"])  # group on two columns
    .agg([
        pl.col("revenue").sum().alias("total_revenue"),
        pl.col("revenue").mean().alias("avg_revenue"),
        pl.col("order_id").count().alias("order_count"),
    ])
    .sort(["category", "region"])     # stable output for readability
)
print("\nAggregation result:")
print(summary)

# --- Write results back to CSV ---
out_path = "/tmp/orders_summary.csv"
summary.write_csv(out_path)
print(f"\nSummary written to {out_path}")

**What just happened?**
- `pl.read_csv()` inferred all types correctly — `revenue` is `Float64`, `quantity` is `Int64`
- `group_by().agg()` accepts a **list of expressions** — you can compute many aggregations in one pass
- `.alias()` renames the output column inline — no separate `.rename()` step needed
- **The whole pipeline is a single chained expression** — Polars executes it in one multi-threaded pass
- `write_csv()` writes the result DataFrame directly to disk — no `to_csv(index=False)` needed

---
## Challenge — do this yourself

Below is a Pandas snippet that does a groupby + filter. Rewrite it in Polars.
Your output DataFrame should have the same columns and values.

```python
# Pandas version
import pandas as pd
df_pd = pd.read_csv("/tmp/orders.csv")
grouped = df_pd.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_qty=("quantity", "mean")
).reset_index()
result = grouped[grouped["total_revenue"] > 600_000]
print(result)
```

In [ ]:
# Your Polars solution here
# Hint: use group_by().agg() then filter()

df = pl.read_csv("/tmp/orders.csv")

result = (
    df
    # TODO: group_by "region"
    # TODO: agg total_revenue = sum of revenue, avg_qty = mean of quantity
    # TODO: filter total_revenue > 600_000
)
print(result)

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Apache Arrow | Column-oriented memory layout — contiguous columns = fast CPU cache use |
| No index | Polars has no row index — row position is always implicit |
| Typed nulls | `None` → `pl.Null` — integers stay integers even with missing data |
| `pl.col()` | All transformations use expressions, not direct column references |
| Immutability | Every operation returns a new DataFrame — no in-place mutation |
| `group_by().agg()` | Multi-expression aggregation in one pass — more concise than Pandas |

> **Tip:** Polars is built on Apache Arrow. The column-oriented memory layout is why it's so fast. Understand this before optimizing.

---
## What's next

**Day 2** → Expressions and the Polars API: `pl.col()`, `pl.lit()`, `pl.when()`, `select()` vs `with_columns()`, and 5 Pandas → Polars translations.

Mark Day 1 complete in your [tracker](../index.html).